# Phase 2 – Data cleaning

This notebook walks through how the raw competition sheets become tidy tables, and shows the evidence behind each cleaning rule. The logic lives in `src/flyball/`, and every rule is also recorded, with row counts, in `ANALYSIS_LOG.md` (Step 1).

**Inputs** (read only, never modified):
- `FLYBALLCZ_versenyadatok.xlsx`: 12 sheets, 10 Czech competitions
- `EFC_versenyadatok.xlsx`: 4 sheets, the 2023 European championship
- `Wild_Runners_Flyball_Team_kutyak_alapadatai.xlsx`: the 51 WRFT dogs

**Outputs** (in `data/processed/`, rebuilt by `scripts/run_phase2_clean.py`):

| File | One row per | Used for |
|---|---|---|
| `competitions.csv` | sheet × day (28) | Q3 per-competition breakdowns, weather join |
| `heats.parquet` | team × heat (8,571) | team results, start/fault analysis |
| `runs.parquet` | dog slot (33,904) | dog speed, passes, lane (Q1), sex/neuter (Q2) |
| `dogs.csv` | WRFT dog (51) | dog attributes |
| `data_quality.csv` | check | this report |
| `wrft_hurdle_check.csv` | WRFT height dog | hurdle-height validation |

**Rule:** rows are never deleted. Every exclusion is a flag, so later analyses can filter, and also test what happens without each rule.

In [1]:
import pandas as pd
from flyball import clean, config, spotcheck

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

tables = clean.build_all()
heats, runs, comps, dogs = tables["heats"], tables["runs"], tables["competitions"], tables["dogs"]
print(f"{len(heats):,} team rows | {heats.heat_id.nunique():,} heats | {heats.race_id.nunique():,} races | "
      f"{len(runs):,} dog slots | {heats.is_wrft.sum():,} WRFT rows")

8,571 team rows | 4,336 heats | 1,172 races | 33,904 dog slots | 1,128 WRFT rows


## 1. Competitions

Each sheet title (cell B1) gives the competition, ring, venue and dates. A 2-day sheet is split into days where the clock drops overnight; the code checks that there is exactly one such drop. EFC sheets each cover one day.

The **Czech championship** title says 3–4 Sep 2023, a Sunday–Monday. A club results page gives **2–3 Sep 2023**, and its best times match this sheet to the hundredth of a second, so the date was corrected (`date_overridden`).

In [2]:
comps[["comp_day_id", "competition", "ring", "venue", "country", "date", "weekday",
       "date_overridden", "n_rows", "n_wrft_rows", "divisions", "first_heat", "last_heat"]]

,comp_day_id,competition,ring,venue,country,date,weekday,date_overridden,n_rows,n_wrft_rows,divisions,first_heat,last_heat
0,S01-D1,Hanácké škvarek,<NA>,Heroltice,CZ,2023-04-22,Saturday,False,109,73,"1,2",10:11,17:17
1,S01-D2,Hanácké škvarek,<NA>,Heroltice,CZ,2023-04-23,Sunday,False,126,88,"1,2",09:06,13:15
2,S02-D1,Eager´s Cup,1,Žamberk,CZ,2023-05-06,Saturday,False,312,64,"1,2,4,8",09:34,18:28
3,S02-D2,Eager´s Cup,1,Žamberk,CZ,2023-05-07,Sunday,False,329,56,"1,2,4,8",08:12,18:07
4,S03-D1,Eager´s Cup,2,Žamberk,CZ,2023-05-06,Saturday,False,329,13,"3,5,6,7",09:24,18:08
5,S03-D2,Eager´s Cup,2,Žamberk,CZ,2023-05-07,Sunday,False,376,8,"3,5,6,7",08:13,18:19
6,S04-D1,Flyvaryors turnaj,<NA>,Blšany,CZ,2023-05-20,Saturday,False,354,37,"1,2,3,4,5",09:08,17:31
7,S04-D2,Flyvaryors turnaj,<NA>,Blšany,CZ,2023-05-21,Sunday,False,386,44,"1,2,3,4,5",08:10,16:42
8,S05-D1,MED's Trophy,<NA>,Pláně,CZ,2023-06-10,Saturday,False,284,0,"1,2,3,4",09:07,16:56
9,S05-D2,MED's Trophy,<NA>,Pláně,CZ,2023-06-11,Sunday,False,295,0,"1,2,3,4",08:08,16:06


## 2. Data-quality checks

Every check the pipeline runs, with the number of rows, slots or dogs affected. `expected` is filled in where an earlier step established the answer.

In [3]:
tables["data_quality"]

,check,n,unit,expected,note
0,rows,8571,rows,8571,
1,wrft_rows,1128,rows,1128,
2,rows_realigned,19,rows,19,Step 1.4
3,heats_paired,4235,heats,,
4,rows_unpaired,101,rows,,opponent row missing or practice
5,paired_heats_same_lane,0,heats,0,
6,paired_heats_both_win,0,heats,0,
7,paired_heats_both_loss,439,heats,,allowed: both teams can lose (UKFL 8.11.M)
8,red_font_mismatch_cells,1,non-empty cells,0,
9,identity_rows,6286,rows,6286,


## 3. Lane colour is hidden in the font colour

There is no lane column. The lane is the **font colour of the team name**: red or blue. If that reading is right, the two teams in every heat must be in different lanes, and they are, in all 4,235 paired heats.

In [4]:
lanes = heats.groupby("sheet", sort=False)["lane"].value_counts().unstack()
lanes["red_share"] = (lanes["red"] / lanes.sum(axis=1)).round(3)
paired = heats[heats.paired]
print("paired heats with one red + one blue team:",
      paired.groupby("heat_id").lane.nunique().eq(2).sum(), "of", paired.heat_id.nunique())
lanes

paired heats with one red + one blue team: 4235 of 4235


lane,blue,red,red_share
sheet,,,
Hanácké škvarek,118,117,0.498
Eager´s Cup 1,319,322,0.502
Eager´s Cup 2,351,354,0.502
Flyvaryors turnaj,371,369,0.499
MED's Trophy,290,289,0.499
BCCCZ mistrovství,194,195,0.501
Crazy Cup,412,412,0.500
"Máchale, spadlo ti to!",334,334,0.500
Mistrovství České republiky,260,261,0.501


## 4. Shifted rows after a "ran in" code

In 19 rows the recorder typed `vběhl` (ran in) into a dog's **changeover** cell, and every later value landed one cell too early. That produced negative "run times" and `ok`/`chyba` codes in run columns. The fix moves the values back one cell. Below is the raw row next to the cleaned one (Křupky na břehu, Excel row 109).

In [5]:
spotcheck.compare(heats, "Křupky na břehu", 109).iloc[:18]

,column,raw,clean,code
0,division,4,4,
1,when,13:08:43,2023-09-16 13:08:43,
2,team,Free X-treme,Free X-treme,
3,opponent,Hopíci Brno,Hopíci Brno,
4,total,19.77,19.77,
5,outcome,P,loss,
6,hurdle_cm,20,20.0,
7,name1,Scotty,Scotty,
8,start,0.07,0.07,number
9,run1,4.98,4.98,number


After realignment, every code sits in the right kind of cell, and no run time is negative (see `test_codes_in_the_right_cells`).

In [6]:
pd.concat({s: heats[f"{s}_code"].value_counts() for s in clean.SLOTS}, axis=1).fillna(0).astype(int)

,start,run1,chg2,run2,chg3,run3,chg4,run4
number,7607,8454,4416,8398,4531,8276,5104,8149
false_start,964,0,0,0,0,0,0,0
outside,0,117,0,64,0,66,0,84
ok,0,0,3151,0,2922,0,2313,0
error,0,0,575,0,550,0,493,0
missing,0,0,281,54,372,166,447,247
early_pass,0,0,148,0,196,0,214,0
run_in,0,0,0,55,0,63,0,91


## 5. Total time identity and reruns

For a heat without reruns: `total = start + 4 runs + 3 changeovers`. Each part is shown to 0.01 s, so 8 rounded parts can be off by up to ±0.04 s. Anything larger and positive is time spent re-running faulted dogs, which has no column of its own.

In [7]:
res = heats["rerun_time"].dropna()
summary = pd.cut(res, [-1, -0.04, 0.04, 1, 5, 100],
                 labels=["< -0.04 (inconsistent)", "±0.04 (identity holds)", "0.04–1 s", "1–5 s", "> 5 s (rerun)"])
summary.value_counts().sort_index().to_frame("rows")

,rows
rerun_time,
< -0.04 (inconsistent),2
±0.04 (identity holds),5588
0.04–1 s,1
1–5 s,44
> 5 s (rerun),651


## 6. Practice starts, incomplete heats, missing results

- **Practice start:** no result, at most one dog, and a total of 8 s or less. Under FCI rules, one dog per team may run once before a team's first races.
- **No result:** heats without a recorded outcome. Their dog times are real, so they are kept.
- **Partial heats:** once a heat is decided, the losing team often stops, so fewer than 4 dogs ran. These are normal.

In [8]:
flags = ["is_practice", "no_outcome", "is_zero_total", "has_placeholder_dog0", "is_complete", "is_clean_heat", "paired"]
heats[flags].sum().to_frame("rows")

,rows
is_practice,28
no_outcome,107
is_zero_total,17
has_placeholder_dog0,14
is_complete,7670
is_clean_heat,5242
paired,8470


## 7. Outlier rules for speed analysis

A dog's run counts as **valid for speed** only if it is a clean numeric run, not a practice run, and passes the plausibility rules. Each rule is its own flag (see the table below for the evidence):

- **< 3.3 s:** faster than the fastest dog ever recorded (3.383 s).
- **> 10 s:** 31 m in 10 s is a trot. These are interruptions.
- **Per-dog rule:** more than max(5 × the dog's robust spread, 1.0 s) slower than its own median, for dogs with at least 10 runs. Most dogs vary by only ~0.1 s, so this catches stumbles, not normal variation.
- **Starts and changeovers > 3 s:** the dog was held back, not passed.

In [9]:
clean_runs = runs[runs.run_code.eq("number") & ~runs.is_practice]
q = [0.001, 0.01, 0.05, 0.5, 0.95, 0.99, 0.999]
pd.DataFrame({
    "run time": clean_runs.run_time.quantile(q),
    "start": runs.loc[runs.position.eq(1) & runs.pass_code.eq("number"), "pass_time"].quantile(q),
    "changeover": runs.loc[runs.position.gt(1) & runs.pass_code.isin(["number", "ok"]), "pass_time"].quantile(q),
}).round(2).rename_axis("quantile")

,run time,start,changeover
quantile,,,
0.001,3.43,0.00,0.00
0.010,3.57,0.00,0.00
0.050,3.76,0.02,0.00
0.500,4.27,0.10,0.16
0.950,5.32,0.28,0.72
0.990,6.54,0.53,2.63
0.999,9.14,8.53,12.18


In [10]:
rules = ["run_fault", "run_too_fast", "run_over_ceiling", "run_dog_outlier", "pass_over_ceiling", "valid_speed", "valid_pass"]
runs[rules].sum().to_frame("dog slots")

,dog slots
run_fault,540
run_too_fast,16
run_over_ceiling,23
run_dog_outlier,249
pass_over_ceiling,213
valid_speed,32967
valid_pass,29809


## 8. WRFT dogs and validations

- WRFT dogs are identified **only inside `WildRunners*` rows**, because opponent teams reuse names like Abby or Chase. 50 of 51 dogs appear; Fibo never ran. No WRFT name is unknown.
- **Hurdle height** should equal the jump height of the team's smallest dog. For a few dogs the recorded hurdle is consistently one step lower (or higher) than their master-data height. The master sheet probably holds a later measurement, so the heat's recorded hurdle stays authoritative.
- **Age:** Hank and Bonnie were younger than the FCI minimum of 18 months at every competition. Their birth dates may be wrong in the master data.

In [11]:
tables["wrft_hurdle_check"].query("share_mismatch > 0")

,dog,jump_height_cm,n_rows_as_height_dog,n_hurdle_lower,n_hurdle_higher,typical_hurdle_cm,share_mismatch
0,Alfie,30.0,35,35,0,27.5,1.000
1,Bailey,32.5,1,1,0,27.5,1.000
2,Brooke,30.0,4,4,0,27.5,1.000
3,Bree,30.0,4,4,0,27.5,1.000
4,Bonnie,27.5,65,65,0,25.0,1.000
5,Henry,20.0,12,0,12,25.0,1.000
6,Maggie,30.0,36,36,0,27.5,1.000
7,Moon,30.0,28,28,0,27.5,1.000
8,Enola,32.5,1,1,0,27.5,1.000
9,Pax,25.0,138,0,61,25.0,0.442


In [12]:
wrft = runs[runs.is_wrft_dog]
(wrft.groupby("dog_key")
     .agg(category=("category", "first"), slots=("row_id", "size"), min_age_months=("age_months", "min"))
     .query("min_age_months < 18"))

,category,slots,min_age_months
dog_key,,,
Bonnie,intact female,105,13.5
Hank,intact male,166,11.7


## 9. Spot-check against the raw Excel

Five heats chosen to cover the tricky cases. Raw values are read straight from the workbook, independently of the pipeline:

1. a WRFT-vs-WRFT heat
2. an EFC heat (English codes, `1 + fraction` clock)
3. a WRFT heat with rerun time
4. a practice row
5. a realigned row (shown in section 4)

In [13]:
for sheet, row in [("Hanácké škvarek", 3), ("EFC_2 ring_1", 3), ("Crazy Cup", 137), ("Eager´s Cup 1", 3)]:
    print(f"=== {sheet}, Excel row {row}")
    print(spotcheck.compare(heats, sheet, row).to_string(index=False), end="\n\n")

=== Hanácké škvarek, Excel row 3


       column                  raw                clean        code
     division                    1                    1            
         when             10:11:57  2023-04-22 10:11:57            
         team WildRunners dreamers WildRunners dreamers            
     opponent WildRunners Maximini WildRunners Maximini            
        total                17.01                17.01            
      outcome                    P                 loss            
    hurdle_cm                   25                 25.0            
        name1                Sunny                Sunny            
        start                -0.24                -0.24 false_start
         run1                 4.21                 4.21      number
        name2                Dixie                Dixie            
         chg2                  0.3                  0.3      number
         run2                 4.24                 4.24      number
        name3                 Alex              

       column                  raw                clean   code
     division                    4                    4       
         when  1900-01-01 08:04:02  2023-07-30 08:04:02       
         team WildRunners dreamers WildRunners dreamers       
     opponent        Jokers ALL-IN        Jokers ALL-IN       
        total                16.76                16.76       
      outcome                    L                 loss       
    hurdle_cm                 27.5                 27.5       
        name1                Sunny                Sunny       
        start                 0.02                 0.02 number
         run1                 4.02                 4.02 number
        name2               Apollo               Apollo       
         chg2                 0.21                 0.21 number
         run2                 4.14                 4.14 number
        name3                  Pax                  Pax       
         chg3                error                  NaN

## 10. What the next phases get

- `runs.query("valid_speed")`: dog speed. Compare **within division**, because lower divisions may hold back to avoid breaking out.
- `runs.query("valid_pass")`: start and changeover skill. `pass_fault` gives the early-pass rate.
- `heats.clean_time`, `has_rerun` and `n_faults`: the team-level picture.
- `lane`, `prev_category` and `comp_day_id`: inputs for Q1, Q2 and Q3.
- `competitions.csv`: venue and date for the weather join.